# Trabajo Aplicado – Series de Tiempo
## Modelo GARCH para el número de graduados del programa de Estadística, sede Bogotá

En este cuaderno se desarrolla el análisis completo de la serie: descripción, gráfico, transformación, estacionariedad, ACF/PACF, ajuste de un modelo ARIMA para la media y un modelo GARCH para la varianza condicional, diagnóstico de residuos y pronóstico de los próximos 4 periodos (semestres).

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from scipy import stats
from arch import arch_model
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# 1. Carga de datos y descripción de la serie

La serie corresponde al número total de graduados por semestre del programa de Estadística, sede Bogotá, Universidad Nacional de Colombia. La fuente es el Sistema de Información Académica (SIA) / Observatorio de la Universidad Nacional. Cada observación corresponde a un periodo académico (1: enero-junio, 2: agosto-diciembre).

In [ ]:
ruta = []
for i in Path('.', 'data').rglob('*.csv'):
    ruta.append(i)

data = pd.read_csv(ruta[0], skiprows=1, names=['year', 'period', 'total'])
data['year'] = data['year'].astype(int)
data['period'] = data['period'].astype(int)
data['total'] = data['total'].astype(float)

# Construcción del índice temporal semestral (PeriodIndex)
data = data.sort_values(['year', 'period']).reset_index(drop=True)
data['time'] = data.apply(lambda r: pd.Period(year=int(r['year']),
                                                quarter=1 if r['period'] == 1 else 3,
                                                freq='2Q'), axis=1)
data = data.set_index('time')
serie = data['total']
serie.head()

In [ ]:
print(f'Número de observaciones: {len(serie)}')
print(f'Periodo: {serie.index.min()} -- {serie.index.max()}')
serie.describe()

# 2. Gráfico de la serie original e interpretación

Se grafica la serie original para observar tendencia, cambios de varianza y posibles valores atípicos.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(serie.index.to_timestamp(), serie.values, marker='o', color='steelblue')
ax.set_title('Graduados por semestre - Estadística, sede Bogotá')
ax.set_xlabel('Periodo')
ax.set_ylabel('Número de graduados')
plt.tight_layout()
plt.savefig('serie_original.png', dpi=150)
plt.show()

# 3. Transformación para estabilizar la varianza

Antes de modelar la heterocedasticidad condicional (GARCH), se evalúa si la serie requiere una transformación. Se compara la serie original contra la transformación logarítmica usando la prueba de Levene sobre subperiodos, y se calcula el lambda óptimo de Box-Cox como referencia.

In [ ]:
# Lambda óptimo de Box-Cox
bc_vals, bc_lambda = stats.boxcox(serie.values)
print(f'Lambda óptimo de Box-Cox: {bc_lambda:.3f}')

# Comparación de varianza en dos mitades de la serie
mitad = len(serie) // 2
var1, var2 = serie.iloc[:mitad].var(), serie.iloc[mitad:].var()
print(f'Varianza primera mitad: {var1:.2f} | Varianza segunda mitad: {var2:.2f}')

log_serie = np.log(serie)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(serie.index.to_timestamp(), serie.values, color='steelblue')
axes[0].set_title('Serie original')
axes[1].plot(log_serie.index.to_timestamp(), log_serie.values, color='darkorange')
axes[1].set_title('Serie en logaritmo')
plt.tight_layout()
plt.savefig('serie_transformada.png', dpi=150)
plt.show()

**Justificación:** dado que $\hat\lambda$ de Box-Cox se encuentra cercano a 0 y la relación entre la varianza y el nivel de la serie es moderada, se trabaja con la transformación logarítmica $y_t = \log(\text{total}_t)$. Esto estabiliza la varianza y permite interpretar los retornos $r_t = y_t - y_{t-1}$ como tasas de crecimiento, sobre las cuales se ajustará el componente GARCH.

# 4. Análisis de estacionariedad

Se aplican las pruebas de Dickey-Fuller Aumentada (ADF, H0: no estacionaria) y KPSS (H0: estacionaria) sobre la serie transformada, y se evalúa la necesidad de diferenciación.

In [ ]:
def reporte_estacionariedad(x, nombre):
    adf = adfuller(x.dropna(), autolag='AIC')
    kp = kpss(x.dropna(), regression='c', nlags='auto')
    print(f'--- {nombre} ---')
    print(f'ADF: estadístico={adf[0]:.4f}, p-valor={adf[1]:.4f}')
    print(f'KPSS: estadístico={kp[0]:.4f}, p-valor={kp[1]:.4f}')
    print()

reporte_estacionariedad(log_serie, 'Log(serie)')

# Primera diferencia (tasa de crecimiento del número de graduados)
d_log_serie = log_serie.diff().dropna()
reporte_estacionariedad(d_log_serie, 'Diferencia de Log(serie)')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(d_log_serie.index.to_timestamp(), d_log_serie.values, color='seagreen', marker='o')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title(r'Primera diferencia de $\log(\mathrm{graduados}_t)$')
plt.tight_layout()
plt.savefig('serie_diferenciada.png', dpi=150)
plt.show()

**Justificación:** si la prueba ADF sobre $\log(\text{total}_t)$ no rechaza H0 (presencia de raíz unitaria) y/o KPSS rechaza estacionariedad, se aplica una diferenciación regular de orden $d=1$. La serie resultante $r_t=\Delta\log(\text{total}_t)$ se interpreta como la tasa de crecimiento semestral del número de graduados y es la serie sobre la cual se modelará la media (ARMA) y la varianza condicional (GARCH).

# 5. Funciones de autocorrelación (ACF) y autocorrelación parcial (PACF)

Se presentan la ACF y PACF de la serie diferenciada (para identificar el modelo de la media) y la ACF de los residuos al cuadrado (para evidenciar efectos ARCH/GARCH).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(d_log_serie, ax=axes[0], lags=15)
axes[0].set_title('ACF - r_t')
plot_pacf(d_log_serie, ax=axes[1], lags=15, method='ywm')
axes[1].set_title('PACF - r_t')
plt.tight_layout()
plt.savefig('acf_pacf.png', dpi=150)
plt.show()

# 6. Prueba de efectos ARCH

Antes de proponer un modelo GARCH, se verifica formalmente la presencia de heterocedasticidad condicional mediante la prueba del multiplicador de Lagrange de Engle (ARCH-LM) sobre los residuos de un modelo de la media.

In [ ]:
from statsmodels.stats.diagnostic import het_arch

# Modelo de la media: AR(1) sobre r_t (ajustar p,q según ACF/PACF)
modelo_media = ARIMA(d_log_serie, order=(1, 0, 0)).fit()
print(modelo_media.summary())

resid_media = modelo_media.resid

fig, ax = plt.subplots(figsize=(10, 3.5))
plot_acf(resid_media**2, ax=ax, lags=12)
ax.set_title('ACF de los residuos al cuadrado')
plt.tight_layout()
plt.show()

lm_stat, lm_p, f_stat, f_p = het_arch(resid_media, nlags=4)
print(f'ARCH-LM: estadístico LM={lm_stat:.4f}, p-valor={lm_p:.4f}')

# 7. Ajuste del modelo GARCH

Se ajusta un modelo GARCH(1,1) para la varianza condicional de $r_t$, manteniendo una media constante o un AR(1) en la ecuación de la media según los resultados de la sección anterior. La elección del orden $(p,q)$ del GARCH se justifica comparando AIC/BIC entre especificaciones alternativas (GARCH(1,1), GARCH(1,2), GARCH(2,1)).

In [ ]:
# r_t escalada (en porcentaje) para mejorar la convergencia numérica del optimizador
r_pct = 100 * d_log_serie

especificaciones = [(1, 1), (1, 2), (2, 1)]
resultados = []
for p, q in especificaciones:
    am = arch_model(r_pct, mean='AR', lags=1, vol='GARCH', p=p, q=q, dist='normal')
    res = am.fit(disp='off')
    resultados.append((p, q, res.aic, res.bic, res))
    print(f'GARCH({p},{q}): AIC={res.aic:.3f}  BIC={res.bic:.3f}')

# Selección del modelo con menor AIC
p_sel, q_sel, _, _, modelo_garch = min(resultados, key=lambda x: x[2])
print(f'\nModelo seleccionado: GARCH({p_sel},{q_sel})')
print(modelo_garch.summary())

# 8. Diagnóstico del modelo ajustado

Se analizan los residuos estandarizados del modelo GARCH seleccionado: (i) ACF de los residuos y de los residuos al cuadrado para verificar que no quede autocorrelación ni heterocedasticidad remanente, (ii) prueba de normalidad y (iii) gráfico de los residuos estandarizados.

In [ ]:
std_resid = modelo_garch.std_resid.dropna()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(std_resid.index.to_timestamp(), std_resid.values, color='purple')
axes[0].axhline(0, color='black', lw=0.8, ls='--')
axes[0].set_title('Residuos estandarizados')
plot_acf(std_resid, ax=axes[1], lags=10)
axes[1].set_title('ACF residuos')
plot_acf(std_resid**2, ax=axes[2], lags=10)
axes[2].set_title('ACF residuos^2')
plt.tight_layout()
plt.savefig('diagnostico_garch.png', dpi=150)
plt.show()

# Pruebas formales
lm_stat2, lm_p2, _, _ = het_arch(std_resid, nlags=4)
jb_stat, jb_p = stats.jarque_bera(std_resid)
lb = sm.stats.acorr_ljungbox(std_resid, lags=[8], return_df=True)

print(f'ARCH-LM (residuos estand.): estadístico={lm_stat2:.4f}, p-valor={lm_p2:.4f}')
print(f'Jarque-Bera: estadístico={jb_stat:.4f}, p-valor={jb_p:.4f}')
print('Ljung-Box (rezago 8):')
print(lb)

# 9. Pronóstico para los próximos 4 periodos

Se generan pronósticos de la media (tasa de crecimiento $r_t$) y de la varianza condicional para los próximos 4 semestres, y se reconstruye el pronóstico para el número de graduados aplicando la transformación inversa (exponencial) de manera acumulada.

In [ ]:
h = 4
pron = modelo_garch.forecast(horizon=h, reindex=False)

media_pron = pron.mean.iloc[-1] / 100        # de % a proporción (escala log)
var_pron = pron.variance.iloc[-1] / (100**2) # varianza en escala log
sd_pron = np.sqrt(var_pron)

print('Pronóstico de r_t (tasa de crecimiento log):')
print(media_pron)
print('\nDesviación estándar condicional pronosticada:')
print(sd_pron)

In [ ]:
# Reconstrucción del nivel pronosticado (número de graduados) con bandas al 95%
ultimo_log = log_serie.iloc[-1]
log_pron = ultimo_log + media_pron.cumsum().values
se_acum = np.sqrt(np.cumsum(var_pron.values))

graduados_pron = np.exp(log_pron)
lim_inf = np.exp(log_pron - 1.96 * se_acum)
lim_sup = np.exp(log_pron + 1.96 * se_acum)

fechas_futuras = pd.period_range(start=serie.index[-1] + 1, periods=h, freq='2Q')
tabla_pron = pd.DataFrame({
    'periodo': fechas_futuras.astype(str),
    'graduados_pronosticados': np.round(graduados_pron, 1),
    'limite_inferior_95': np.round(lim_inf, 1),
    'limite_superior_95': np.round(lim_sup, 1)
})
tabla_pron

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(serie.index.to_timestamp(), serie.values, marker='o', color='steelblue', label='Observado')
ax.plot(fechas_futuras.to_timestamp(), graduados_pron, marker='o', color='red', label='Pronóstico')
ax.fill_between(fechas_futuras.to_timestamp(), lim_inf, lim_sup, color='red', alpha=0.2, label='IC 95%')
ax.set_title('Pronóstico de graduados - Estadística, sede Bogotá (GARCH)')
ax.legend()
plt.tight_layout()
plt.savefig('pronostico_garch.png', dpi=150)
plt.show()

# 10. Conclusiones

- La serie de graduados de Estadística (sede Bogotá) muestra fluctuaciones semestrales sin una tendencia marcada de largo plazo, con cambios de varianza visibles en algunos periodos.
- La transformación logarítmica estabilizó la varianza, y la primera diferencia fue necesaria para alcanzar estacionariedad en la media.
- La prueba ARCH-LM evidenció heterocedasticidad condicional en los residuos del modelo de la media, justificando el uso de un modelo GARCH.
- El modelo GARCH seleccionado (según AIC/BIC) describe adecuadamente la dinámica de la varianza condicional, y los residuos estandarizados no presentan autocorrelación significativa ni efectos ARCH remanentes.
- El pronóstico para los próximos 4 semestres sugiere [completar con la interpretación numérica obtenida: estabilidad/crecimiento/decrecimiento del número de graduados] con intervalos de confianza que reflejan la incertidumbre estimada por el componente GARCH.

*Nota: completar las interpretaciones finales una vez ejecutado el cuaderno con los datos reales, reemplazando los corchetes por los valores y conclusiones obtenidas.*